# GMRESについて

## 概要 (後で書く)

### Krylov部分空間

解く問題は $A x = b$ という連立一次方程式である。<br>
ここで A は $n \times n$ 次元の行列、$x, b$ はそれぞれ $n$ 次元のベクトルとする。

GMRESでは近似初期解を $x_0$ として 近似解を $x_m = x_0 + z_m$ としたときにこの$z_m$ をKrylov部分空間 $\mathcal{K}_m = \text{span}\{r_0, A r_0, A^2 r_0, \cdots A^{(m-1)} r_0\}$ の中で繰り返し計算で探すことで、元の連立一次方程式を効率よく解く。


このKrylov部分空間だけを見てみても連立方程式を解くことイメージが繋がりにくいので、この先の Arnoldi 法でつながりを見ていく。

###  Arnoldi法

#### Step 0

近似初期解を$x_0$ として 初期残差を $r_0 = b - Ax_0$ とする。<br>
$v_1 = r_0 / ||r_0||$ として、$r_0$ 方向の単位ベクトル $v_1$ を作る

このとき $K_1= \text{span}\{v_1 \} = \text{span}\{r_0\}$<br>
よって、$v_1$ がKrylov部分空間の基底になる。

#### Step 1

$h_{11} = v_1^T A v_1$ として、$A v_1$ と $v_1$ の内積から、$A v_1$が基底 $v_1$にどれだけ乗っているかを計算する。ここで$A v_1$ はベクトル $v_1$ に線形変換 $A$ を作用させて得られる新しいベクトルである。

$w = A v_1 - h_{11} v_1$ として、$A v_1$方向から$v_1$を引き、$v_1$に直交するベクトルを作成する。

$h_{21} = ||w||$ として、$v_1$に直交するベクトルの大きさを求める。

$v_2 = w / h_{21}$ として、$v_1$に直交する単位ベクトル $v_2$ を求める。

このとき 新たな基底として$v_2$を採用し $K_2= \text{span}\{v_1 , v_2\}$<br>
よって、$v_1, v_2$がKrylov部分空間の基底になる。

Arnoldi法では、各ステップで $A v_j$ を既存の基底と新しい基底の線形結合で表す。このとき表れる係数 $h_{ij}$ を列ごとに並べた行列が上ヘッセンベルグ行列 $H$ である。

$$
\boxed{
A v_1 = h_{11} v_1 + h_{21} v_2
}
$$

この係数を一列目に並べると、

$$
\bar{H}_1 = \begin{bmatrix}
h_{11} \\ h_{21}
\end{bmatrix}
$$

-------------------------------------------------------------

##### Krylov部分空間 $\{r_0 , A r_0 \}$ との関係

部分空間 $\text{span}$ の定義は
$\text{span}\{x_1, x_2\}= \{c_1 x_1 + c_2 x_2 | c_1 , c_2 \in \R\}$であり、$\text{span}\{x_1, x_2\}$ は任意の $c1,c2$ を係数としたベクトル $c_1 x_1 + c_2 x_2$ の線形結合の集合である。

集合が$A,B$ 二つあり、$A=B$を示すには、 $A \subseteq B$ と $B \subseteq A$ を示す必要がある。

ここでは、$\text{span}\{v_1, v_2\}$ = $\text{span}\{r_0, A r_0 \}$を示す。

##### $\text{span}\{v_1, v_2\}$ $\subseteq$ $\text{span}\{r_0, A r_0 \}$ を示す

$\alpha = 1/||r_0||$とすると、$v_1 = \alpha r_0$である。

また、$v_2 = (A v_1 - h_{11} v_1)/h_{21}$ であり、$\beta = - \alpha h_{11}/h_{21}, \gamma = \alpha / h_{21}$ とすると、$v_2 = \beta r_0 + \gamma A r_0$になる。<br>
任意のベクトルは 任意の $c_1, c_2$に対して $c_1 v_1 + c_2 v_2$ と書ける。<br>
$v_1,v_2$を代入すると以下となる。

$$
c_1 \alpha r_0 + c_2 (\beta r_0 + \gamma A r_0) = (c_1 \alpha + c_2 \beta) r_0 + c_2 \gamma A r_0
$$

よって $c_1 v_1 + c_2 v_2 \in \text{span}\{ r_0, A r_0\}$ であり、

$$
\text{span}\{v_1, v_2\} \subseteq \text{span}\{r_0, A r_0 \}
$$

##### $\text{span}\{r_0, A r_0\}$ $\subseteq$ $\text{span}\{v_1, v_2 \}$ を示す

$v_2 = (A v_1 - h_{11} v_1)/h_{21}$より、$A v_1 = h_{11} v_1 + h_{21} v_2$ となる。<br>
$v_1 = \alpha r_0$ より、$A v_1 = \alpha A r_0$ である。
よって $A r_0 = (h_{11} v_1 + h_{21} v_2) /\alpha$ となり $A r_0$ は $v_1, v_2$の線形結合で表すことが出来る。
任意のベクトルは任意の $d_1, d_2$ に対して $d_1 r_0 + d_2 A r_0$ と書ける。<br>
$r_0, A r_0$ を代入すると以下となる。

$$
d_1 r_0 + d_2 A r_0 = \frac{d_1}{\alpha} v_1 + \frac{d_2 (h_{11} v_1 + h_{21} v_2)}{\alpha} =  \left(\frac{d_1}{\alpha} +\frac{d_2 h_{11}}{\alpha}\right) v_1 + \frac{d_2 h_{21}}{\alpha} v_2 
$$

よって、$d_1 r_0 + d_2 A r_0 \in \text{span} \{v_1, v_2\}$であり、

$$
\text{span}\{r_0, A r_0\} \subseteq \text{span}\{v_1, v_2 \}
$$

以上より、以下を示せる。

$$
\text{span}\{v_1, v_2\} = \text{span}\{r_0, A r_0 \}
$$

-------------------------------------------------------------

### Step 2

$h_{12} = v_1^T A v_2$ として、$A v_2$と $v_1$ の内積から、$A v_2$が基底$v_1$にどれだけ乗っているかを計算する。<br>
$h_{22} = v_2^T A v_2$ として、$A v_2$と $v_2$ の内積から、$A v_2$が基底$v_2$にどれだけ乗っているかを計算する。<br>
ここで$A v_2$ ベクトル $v_2$ に線形変換$A$を作用させて得られる新たなベクトルである。

$w = A v_2 - h_{12} v_1 - h_{22} v_2$ として、 $A v_2$から$v_1, v_2$要素を引き、$v_1, v_2$に直交するベクトルを作成する。

$h_{32} = ||w||$ として、$v_1, v_2$に直交するベクトルの大きさを求める。<br>
$v_3 = w / h_{32}$ として、$v_1, v_2$に直交する単位ベクトル $v_3$ を求める。

このとき 新たな基底として $v_3$を採用し、$K_3= \text{span}\{v_1 , v_2, v_3\}$<br>
よって、$v_1, v_2, v_3$がKrylov部分空間の基底になる。

このStepで得られる $A v_2$の展開は以下となる。

$$
\boxed{
A v_2 = h_{12} v_1 + h_{22} v_2 + h_{32} v_3
}
$$

この式の係数を上ヘッセンベルグ行列の第２列に追加すると

$$
\bar{H}_2 = \begin{bmatrix}
h_{11} & h_{12} \\
h_{21} & h_{22} \\
0 & h_{32}
\end{bmatrix}
$$

ここでも Step 1 と同様に $\text{span} \{ v_1, v_2, v_3\} = \text{span} \{r_0, A r_0, A^2 r_0 \}$ と示せるが割愛する。

ここでこれまで得られた式を再掲すると以下となる。

$$
\begin{aligned}
A v_1 &= h_{11} v_1 + h_{21} v_2 \\
A v_2 &= h_{12} v_1 + h_{22} v_2 + h_{32} v_3
\end{aligned}
$$

これを横に並べると、左辺は

$$
\begin{bmatrix} A v_1 & A v_2 \end{bmatrix} = A \begin{bmatrix} v_1 & v_2 \end{bmatrix} 
$$

右辺は以下となる。

$$
\begin{bmatrix}
h_{11} v_1 + h_{21} v_2 & h_{12} v_1 + h_{22} v_2 + h_{32} v_3
\end{bmatrix} = 
\begin{bmatrix}v_1 & v_2 & v_3 \end{bmatrix}
\begin{bmatrix}h_{11} & h_{12} \\ h_{21} & h_{22} \\ 0 & h_{32}  \end{bmatrix}
$$

ここで$V_2 = \begin{bmatrix} v_1 & v_2 \end{bmatrix},V_3 = \begin{bmatrix} v_1 & v_2 & v_3 \end{bmatrix}$ また$\bar{H}_2$を以下とすると、

$$
\bar{H}_2 = 
\begin{bmatrix}h_{11} & h_{12} \\ h_{21} & h_{22} \\ 0 & h_{32}  \end{bmatrix}
$$

$$
\boxed{
A V_2 = V_3 \bar{H}_2
}
$$

#### Arnoldi法の一般化

初期近似解と $x_0$ と初期残差 $r_0$ を設定し、Arnoldi法を$m$ ステップまで実行する。<br>
第$j$ステップ ($j = 1,\cdots, m$)では、次の計算を行う。

$$
\begin{aligned}
h_{ij} &= v_i^T A v_j \space , \space (i = 1,\cdots , j)\\
w &= A v_j - \sum_{i=1}^j h_{ij} v_i \\
h_{j+1,j} &= ||w|| \\
v_{j+1} &= \frac{w}{h_{j+1, j}}
\end{aligned}
$$

これを$j=1,\cdots , m$ について繰り返す。

そしてKrylov部分空間は $\text{span}\{v_1, \cdots , v_{m+1}\} = \text{span}\{r_0 , \cdots , A^m r_0\}$ となる。

また、上ヘッセンベルグ行列は以下となる。

$$
\bar{H}_m = 
\begin{bmatrix}
h_{11} & h_{12} & h_{13} & \cdots & h_{1m} \\
h_{21} & h_{22} & h_{23} & \cdots & h_{2m} \\
0 & h_{32} & h_{33} & \cdots & h_{3m} \\
0 & 0 &  h_{43} & \cdots & h_{4m} \\
\vdots & \vdots & \ddots & \ddots & \vdots \\
0 & 0 & \cdots & h_{m,m-1} & h_{mm} \\
0 & 0 & \cdots & 0 & h_{m+1, m}
\end{bmatrix}
$$

$\bar{H}_m$の第$j$列には、$A v_j = \sum_{i=1}^{j+1} h_{ij}v_i$ の係数 $h_{ij}$が格納される。$A v_j$ の展開には$v_{j+1}$より後の基底は現れないため、第$j$列の $j+2$ 行目以降はゼロとなる。この構造が上ヘッセンベルグ形である。

各ステップで得られた $A v_j = \sum_{i=1}^{j+1} h_{ij} v_i$ を横に並べて整理すると、以下の関係を得る。

$$
\boxed{
A V_m = V_{m+1} \bar{H}_m
}
$$

$\bar{H}_m$ は $(m+1) \times m$ の行列であることに注意する。


Arnoldi法で生成される基底 $v_1, v_2 , \cdots$ はすべて $\R^n$ のベクトルであり、互いに正規直交する。$n$次元空間内で線形独立なベクトルは最大でも$n$本であるため、Krylov部分空間の次元およびArnoldi法の反復回数の上限は $n$ 回である。

ただし途中で

$$
h_{j+1, j} = 0
$$ 

となった場合、それ以上の独立な基底を生成できなくなったことを意味し、計算は打ち切られる。

実際の大規模問題では計算量を抑えるため、通常は $m \ll n$ として打ち切る。

Arnoldi法は、残差 $r_0$ から始めて、行列 $A$ を作用させることでKrylov部分空間を少しずつ拡張し、その空間の正規直交基底 $V_m$ を生成する方法である。同時に、$A$ をこの基底上で表現した小さな行列 $\bar{H}_m$ が得られ、

$$
AV_m = V_{m+1}\bar{H}_m
$$

という関係が成立する。GMRESでは、この関係を利用して、元の $n$ 次元の問題を $m$ 次元程度の小さな最小二乗問題へ変換する。

### 近似解を求めるための最小二乗問題

Arnoldi法により、初期近似解 $x_0$ から Krylov部分空間での正規直交基底 $V_m, V_{m+1}$ と 上ヘッセンベルグ行列 $\bar{H}_{m}$ が得られた。この状態から $A x = b$ の近似解 $x_m$を求める。

GMRESは繰り返し計算より解を近似的に求める。よって、Arnoldi法 $m$ 回目の近似解 $x_m$ は以下のように更新される。

$$
x_m = x_0 + z_m
$$

この近似解 $x_m$ が得られた場合の残差 $r$ は次のようになる。

$$
r = b - A x_m = b - A(x_0 +z_m) = b - A x_0 - A z_m = r_0 - A z_m
$$

$z_m$ は Krynov部分空間で表現される$x_0$から解候補 $x$ までの更新量であり、$z_m \in \text{span}\{v_1, \cdots , v_m\}$ である。これは、$z_m = y_1 v_1 + \cdots y_m v_m$ のように$z_m$ 表されることを示している。<br>
$Y=[y_1, \cdots, y_m]^T$ と置けば、$z_m = V_m Y$ と表現できる。この$Y$ が求めるべき値である。

Arnoldi法で得られた $AV_m = V_{m+1} \bar{H}_m$ より

$$
r = r_0 - A V_m Y = r_0 - V_{m+1} \bar{H}_m Y  
$$

ここで $\beta = ||r_0||$ とすると、 $v_1 = r_0 / \beta$ より $r_0 = \beta v_1$となる。

また、$e_1 = [1, 0, \cdots, 0]^T \in \mathcal{R}^{m+1}$ とすると、$v_1$ は次のようになる。

$$
v_1 = V_{m+1} e_1 = [v_1, \cdots, v_{m+1}]\begin{bmatrix} 1 \\ 0 \\ \vdots \\ 0 \end{bmatrix}
$$

よって 残差 $r$ は次のようになる。

$$
r = \beta V_{m+1} e_1 - V_{m+1} \bar{H}_m Y
$$

$\beta$ はスカラーであるので、次のようにまとめられる。

$$
r =  V_{m+1} \left(\beta e_1 - \bar{H}_m Y \right)
$$

この残差の大きさ $||r||^2 = r^T r$ を近似解 $x_m$ により最小化したいという問題になる。

$V_{m+1}$ は正規直交基底であるたため、$V_{m+1}^T V_{m+1} = \mathbf{I}$ となる。よって以下となる。

$$
\begin{aligned}
||r||^2 =& \left(V_{m+1} \left(\beta e_1 - \bar{H}_m Y \right)\right)^T\left(V_{m+1} \left(\beta e_1 - \bar{H}_m Y \right)\right)\\
=& \left(\beta e_1 - \bar{H}_m Y \right)^T V_{m+1}^T V_{m+1} \left(\beta e_1 - \bar{H}_m Y \right) \\
=&  \left(\beta e_1 - \bar{H}_m Y \right)^T \left(\beta e_1 - \bar{H}_m Y \right) \\
=& ||\beta e_1 - \bar{H}_m Y ||^2
\end{aligned}
$$

よって2ノルム($||x|| = \sqrt{\sum_i x_i^2}$) は

$$
||r|| = ||\beta e_1 - \bar{H}_m Y ||
$$

求めたいものは $Y = [y_1, \cdots, y_m]^T$ であり、この$Y$ による最小化を考える。

$$
\boxed{
\min\limits_Y ||\beta e_1 + \bar{H}_m Y||
}
$$

ここで $Y$ が求まれば、$z_m = V_m y$ と $x_m = x_0 + z_m$ より近似解 $x_m$ を求めることが出来る。

この最適化問題は未知数が $Y \in \mathcal{R}^m$ のみであり、元のn次元の連立一次方程式は、$m$次元の小さな最小化問題へ変換されたことになる。


### Y の計算

以下の最小化問題であるため、

$$
\min\limits_Y ||\beta e_1 - \bar{H}_m Y||
$$

最小化する式をゼロとすると、

$$
\beta e_1 - \bar{H}_m Y = 0 \rightarrow \bar{H}_m Y =  \beta e_1
$$

のようになるが、$\bar{H}_m$ は $(m+1) \times m$ の縦長行列であるため、疑似逆行列を用いて解くこともできる。<br>
しかしGMRESでは、計算量を押さえながら逐次的に更新できるGives回転を用いて解く。

### QR分解

行列 $A$ を正規直交行列 $Q$ と 上三角行列 $R$ に分解することを QR分解という。

$$
A = QR
$$

正規直交行列は $Q^T Q = \mathbf{I}$ となる性質がある。よって、

$$
Q^T A = R
$$

とすると、上三角行列 $R$ を求めることが出来る。

これを残差2ノルムの式に当てはめて考える。

$$
||r|| = ||\beta e_1 - \bar{H}_m Y ||
$$

今 $\bar{H}_m = QR$ と分解したときに、$Q^T \bar{H}_m = R$ となる関係がある。<br>
$Q^T Q= \mathbf{I}$ であるため、以下のように2ノルムを変化させない

$$
||Qx||^2 = (Qx)^T(Qx) = x^T Q^T Q x = x^T x = ||x||^2 \rightarrow ||Qx|| = ||x||
$$ 

上記のように直交行列 $Q$をベクトルに掛けても2ノルムは変化しない。よって、

$$
||Q^T \ r|| = ||Q^T\left(\beta e_1 - \bar{H}_m Y\right) || = ||Q^T \beta e_1 - R Y ||
$$

これより、以下の最小化問題となり、

$$
\min\limits_Y ||Q^T \beta e_1 - R Y ||
$$

ここで、$Q^T \beta e_1 = g$として、次のように解くことが出来る。

$$
R Y = g
$$

ここで、$R \in \mathcal{R}^{(3,2)}$の上三角行列、$Y =[y_1, y_2]^T$, $g = [g_1, g_2, g_3]^T$ とすると、以下の式となり、

$$
\begin{bmatrix}
r_{11} & r_{12} \\
0 & r_{22} \\
0 & 0 
\end{bmatrix}
\begin{bmatrix}
y_1 \\ y_2
\end{bmatrix}
=
\begin{bmatrix}
g_1 \\ g_2 \\ g_3
\end{bmatrix}
$$

上2行から後退代入で解($y_1, y_2$)が求める。

$$
\begin{aligned}
r_{11} y_1 + r_{12} y_2 &= g_1 \\
r_{22} y_2 &= g_2
\end{aligned} \rightarrow
\begin{aligned}
y_2 &= g_2 / r_{22} \\
y_1 &= \left(g_1 - g_2 r_{12}/r_{22} \right) / r_{11}  \\
\end{aligned}
$$

後退代入により上2行は厳密に満たされるため、最後の1行だけが残差になる。

$$
\min\limits_Y ||g - R Y || = \min\limits_Y || [0 , 0 , g_3]^T || = \sqrt{g_3 \times g_3} = |g_3|
$$

従って、残差2ノルム $||r||$ は$|g_3|$ を評価すれば良い。

このようにQR分解によって$\bar{H}$は上三角行列に変換されるため、求めたい $Y$ を後退代入によって容易に求めることが出来る。しかし、ステップ毎にQR分解を毎回最初から計算すると計算量が大きい。そこでGMRESでは、Givens回転を用いて、各ステップでQR分解を逐次更新する。


### Givens回転

各ステップでGivens回転を行う前に、Givens回転行列とは何かを説明する

Givens回転は以下のようなもので、$k$ は第k列の対角線直下の要素をゼロにするための回転を表す。空白の箇所はゼロである。

$$
G_k = 
\begin{bmatrix}
I_{k-1} &  &\\ 
& c & s& \\
& -s & c& \\
& & &  I 
\end{bmatrix}
$$

例えば、$H=[h_1, h_2]^T$ という行列をから$h_2$を消去することを考える。

ここで、$l = \sqrt{h_1^2 + h_2^2}$ とすると、座標($h_1, h_2$)から、$h_1$は直角三角形の底辺(X方向)、$h_2$は高さ(Y方向)とみなせる。このとき、

$$
c = \cos(\theta) = \frac{h_1}{l}, \quad s = \sin(\theta) = \frac{h_2}{l}
$$

となるような回転角 $\theta$ を選び、次の$G$ を $H$ に掛ける。

$$
G = \begin{bmatrix}
c & s \\
-s & c
\end{bmatrix}
$$

するとこのように、以下のように$H$を$G$で回転することになる。$G$は時計回りに$H$を回転させるため、直角三角形の高さ(Y方向)が消え、X方向が残る。

$$
G H= \begin{bmatrix}
c & s \\
-s & c
\end{bmatrix}
\begin{bmatrix}
h_1 \\ h_2
\end{bmatrix} =
\begin{bmatrix}
c h_1 + s h_2 \\
-s h_1 + c h_2
\end{bmatrix} =
\begin{bmatrix}
 \dfrac{h_1^2 + h_2^2}{l} \\
- \dfrac{h_1 h_2}{l}  + \dfrac{h_1 h_2}{l}
\end{bmatrix} 
=\begin{bmatrix}
l \\ 0
\end{bmatrix}
$$

これがGivens回転の基本的な考え方である。


$\bar{H}_2$ をGivens回転して、上三角行列 $R$ を作ることを考える。

$$
\bar{H}_2 = \begin{bmatrix}
h_{11} & h_{12} \\
h_{21} & h_{22} \\
0 & h_{32}
\end{bmatrix}
$$

Step1 で 1列目の$h_{21}$ を消去し、Step 2 で 2列目の$h_{32}$ を消去する。

Step1 

さっきの直角三角形の考え方で、$h_{11}$ を底辺(X方向)、$h_{21}$ を高さ(Y方向)とする。

すると、$l_1 = \sqrt{h_{11}^2 + h_{21}^2}$として、次の $G_1$ を$\bar{H}_2$ の左側からかける。

$$
G_1 = 
\begin{bmatrix}
h_{11} /l_1 & h_{21} / l_1 & 0\\
- h_{21} /l_1 & h_{11} / l_1 & 0 \\
0 & 0 & 1
\end{bmatrix}
$$

すると、次のように1列目の $h_{21}$ が消える。また2列目も同時に回転するため、$h \rightarrow g$ としている。

$$
G_1 \bar{H}_2 = 
\begin{bmatrix}
h_{11} /l_1 & h_{21} / l_1 & 0\\
- h_{21} /l_1 & h_{11} / l_1 & 0\\
0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
h_{11} & h_{12} \\
h_{21} & h_{22} \\
0 & h_{32}
\end{bmatrix} =
\begin{bmatrix}
l_1 & g_{12} \\
0 & g_{22} \\
0 & g_{32}
\end{bmatrix} 
$$

Step 2

次に2列目の $g_{32}$ を消去することを考える。

($g_{22}, g_{32}$) を直角三角形の底辺と高さと考え、$l_2 = \sqrt{g_{22}^2 + g_{32}^2}$ を長さとする。

次の $G_2$ により、$G_1 \bar{H}_2$ を更に回転させる。

$$
G_2 = 
\begin{bmatrix}
1 & 0 & 0 \\
0 & g_{22} /l_2 & g_{32} / l_2 \\
0 & - g_{32} /l_2 & g_{22} / l_2 \\
\end{bmatrix}
$$

すると、次のように$g_{32}$ が消えて、上三角行列が現れる。

$$
G_2 G_1 \bar{H}_2 = 
\begin{bmatrix}
1 & 0 & 0 \\
0 & g_{22} /l_2 & g_{32} / l_2 \\
0 & - g_{32} /l_2 & g_{32} / l_2 \\
\end{bmatrix}
\begin{bmatrix}
l_1 & g_{12} \\
0 & g_{22} \\
0 & g_{32}
\end{bmatrix} 
=
\begin{bmatrix}
l_1 & g_{12} \\
0 & l_2 \\
0 & 0
\end{bmatrix}
$$

ここで、次のQR分解の関係があり、Givens回転によりQR分解ができることが分かる。

$$
G_2 G_1 H = R \rightarrow Q^T H = R
$$

もし、毎回最初からQR分解を行えば計算量が大きい。そこで、GMRESでは、前のステップで得られたGivens回転を再利用し、新しく追加された列に対して1回だけGivens回転を追加することでQR分解を逐次更新する。

### GMRESの流れ

$A x = b$ の方程式を解く問題を考える。ここで$A \in \mathcal{R}^{n \times n}$である。

Arnoldi法で $m$ 回までStepを繰り返すとする。

#### Step 0
初期近似解 $x_0$ として、$r_0 = b - A x_0$ を計算する。

$$
\boxed{
\beta = ||r_0|| 
}
$$

$$
\boxed{
v_1 = r_0 / \beta
}
$$

#### Step 1

Arnoldi法 1回目の処理で次を実行する。

$$
\begin{aligned}
h_{11} &= v_1 ^ T A v_1 \\
w &= A v_1 - h_{11} v_1 \\
h_{21} &= ||w|| \\
v_2 &= w / h_{21}
\end{aligned}
$$

次に上ハイゼンベルグ行列を得る。

$$
\bar{H}_1 =
\begin{bmatrix}
h_{11} \\ h_{21}
\end{bmatrix}
$$

Givens回転前の右辺を次のようにする。

$$
g^{(0)} = \beta e_1 = \beta \begin{bmatrix} 1 \\ 0 \end{bmatrix}
$$

Givens回転の係数を次のようにし、

$$
\begin{matrix}
l_1 = \sqrt{h_{11}^2 + h_{21}^2} \ ,& c_1 = h_{11} / l_1 \ ,& s_1 = h_{21}/ l_1
\end{matrix}
$$

Givens回転行列を次のように構成し、

$$
G_1 = \begin{bmatrix}
c_1 & s_1 \\
-s_1 & c_1
\end{bmatrix}
$$

$\bar{H}_1$ へかける。

$$
\bar{R}_1 = G_1 \bar{H}_1 = 
\begin{bmatrix}
l_1 \\ 0
\end{bmatrix}
$$

右辺も$G_1$をかけ回転させる。

$$
g^{(1)} =G_1 g^{(0)} = \beta \begin{bmatrix}
c_1 \\ -s_1
\end{bmatrix}
$$

残差 $||r_1|| = |\beta s_1|$ を確認して、小さければここでArnoldi法を終了させ、次の最小化問題より$y_1$を求める。
残差が大きい場合次のStepへ進む。

$$
\min\limits_{y_1} || g^{(1)} - \bar{R}_1 y_1||
$$

$$
g^{(1)} - \bar{R}_1 y_1 = \beta \begin{bmatrix}
c_1 \\ -s_1
\end{bmatrix} - \begin{bmatrix}
l_1 \\ 0
\end{bmatrix}
y_1 = 0
$$

より、$y_1 = \beta c_1 /l_1$となって、近似解$x = x_0 + v_1 y_1$ となる。

今 m=1 であるため、$V_m = [v_1]$ である。 

#### Step 2

Arnoldi法 2回目の処理で次を実行する。

$$
\begin{aligned}
h_{12} &= v_1 ^ T A v_2 \\
h_{22} &= v_2 ^ T A v_2 \\
w &= A v_2 - h_{12} v_1 - h_{22} v_2 \\
h_{23} &= ||w|| \\
v_3 &= w / h_{23}
\end{aligned}
$$


次に上ハイゼンベルグ行列を得る。


$$
\bar{H}_1 =
\begin{bmatrix}
h_{11} \\ h_{21}
\end{bmatrix}
$$

Givens回転前の右辺を次のようにする。

$$
g^{(0)} = \beta e_1 = \beta \begin{bmatrix} 1 \\ 0 \end{bmatrix}
$$

Givens回転の係数を次のようにし、

$$
\begin{matrix}
l_1 = \sqrt{h_{11}^2 + h_{21}^2} \ ,& c_1 = h_{11} / l_1 \ ,& s_1 = h_{21}/ l_1
\end{matrix}
$$

Givens回転行列を次のように構成し、

$$
G_1 = \begin{bmatrix}
c_1 & s_1 \\
-s_1 & c_1
\end{bmatrix}
$$
